In [1]:
# Make helpers.py importable, whatever folder Jupyter started the kernel in
import sys
from pathlib import Path

_candidates = [Path.cwd(), Path.cwd() / "code",
               *Path.cwd().glob("*/code"), *Path.cwd().glob("*/*/code")]
CODE = next(p for p in _candidates if (p / "helpers.py").is_file())
sys.path.insert(0, str(CODE))

# Generated with Claude (Anthropic): download the number of ReliefWeb reports
# per country and year, or use the saved counts if no appname is available.
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv

from helpers import find_root

ROOT = find_root()
RAW = ROOT / "data" / "raw"
OUT = RAW / "reliefweb_counts.csv"
URL = "https://api.reliefweb.int/v2/reports"
FIRST_YEAR, LAST_YEAR = 1999, 2025

load_dotenv(ROOT / ".env")
APPNAME = os.getenv("RELIEFWEB_APPNAME")
DOWNLOAD = APPNAME is not None
print("Repository folder:", ROOT)
print("Download from the API:", DOWNLOAD)

Repository folder: /files/project/humanitarian-underfunding
Download from the API: True


In [2]:
def count_reports_by_year(iso3):
    """Number of ReliefWeb reports with this primary country, per year."""
    query = {
        "filter": {
            "operator": "AND",
            "conditions": [
                {"field": "primary_country.iso3", "value": iso3.lower()},
                {"field": "date.original",
                 "value": {"from": f"{FIRST_YEAR}-01-01T00:00:00+00:00",
                           "to": f"{LAST_YEAR}-12-31T23:59:59+00:00"}},
            ],
        },
        "facets": [{"field": "date.original", "interval": "year"}],
        "limit": 0,
    }
    r = requests.post(URL, params={"appname": APPNAME}, json=query, timeout=60)
    r.raise_for_status()
    return r.json()

In [3]:
if DOWNLOAD:
    test = count_reports_by_year("YEM")
    years = test["embedded"]["facets"]["date.original"]["data"]
    print("Total:", test["totalCount"], "| Sum of years:", sum(y["count"] for y in years))

Total: 21122 | Sum of years: 21122


In [4]:
if DOWNLOAD:
    fts = pd.read_csv(RAW / "fts_globalcluster.csv")
    countries = sorted(fts["countryCode"].dropna().unique())
    print(len(countries), "countries")

    rows = []
    for iso3 in countries:
        result = count_reports_by_year(iso3)
        for item in result["embedded"]["facets"]["date.original"]["data"]:
            rows.append({"countryCode": iso3,
                         "year": int(item["value"][:4]),
                         "reports": item["count"]})
        time.sleep(1)  # be polite to the API

    counts = pd.DataFrame(rows)

    # Years with no report may be missing from the answer: add them with 0
    full = pd.MultiIndex.from_product(
        [countries, range(FIRST_YEAR, LAST_YEAR + 1)],
        names=["countryCode", "year"])
    counts = (counts.set_index(["countryCode", "year"])
                    .reindex(full, fill_value=0)
                    .reset_index())

    counts.to_csv(OUT, index=False)
    print("Saved", OUT, counts.shape)
else:
    print("No appname: using the saved file", OUT)

84 countries
Saved /files/project/humanitarian-underfunding/data/raw/reliefweb_counts.csv (2268, 3)
